<a href="https://colab.research.google.com/github/Lornatt88/maai902-nlp-solution/blob/main/notebooks/02b_long_sentences.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 2b: Shortening long sentences
Some sentences are longer than BERT's 128-token limit. Instead of cutting off their ending, an LLM picks the most important parts, so the meaning is kept.

## 1. Counting long sentences
Counts how many sentences in each split are longer than 128 tokens, using Legal-BERT's tokenizer (the model we will train).

In [1]:
!pip install -q datasets transformers

from datasets import load_dataset
from transformers import AutoTokenizer

ds = load_dataset("coastalcph/lex_glue", "unfair_tos")
tok = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")

for split in ["train", "validation", "test"]:
    lengths = [len(tok(t)["input_ids"]) for t in ds[split]["text"]]      # length of every sentence
    long = sum(1 for n in lengths if n > 128)                              # how many are too long
    print(f"{split}: {long} sentences longer than 128 tokens (longest: {max(lengths)})")

README.md:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

unfair_tos/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  501kB            

unfair_tos/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

unfair_tos/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  147kB            

unfair_tos/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

train: 71 sentences longer than 128 tokens (longest: 478)
validation: 39 sentences longer than 128 tokens (longest: 288)
test: 18 sentences longer than 128 tokens (longest: 331)


Using Legal-BERT’s tokenizer, 71 training, 39 validation and 18 test sentences are longer than 128 tokens (128 in total, about 1.4% of the data). These are the sentences that will be shortened.

## 2. Collecting the long sentences
Gathers every sentence longer than 128 tokens and remembers which split and position it came from, so the shortened version can be put back in the right place.

In [2]:
long_items = []    # (split, position, text) for every long sentence

for split in ["train", "validation", "test"]:
    for i, t in enumerate(ds[split]["text"]):
        if len(tok(t)["input_ids"]) > 128:
            long_items.append((split, i, t))

print("Long sentences collected:", len(long_items))
print("\nExample:", long_items[0][2])

Long sentences collected: 128

Example: to the fullest extent permitted by applicable law , in no event will tinder , its affiliates , employees , licensors or service providers be liable for any indirect , consequential , exemplary , incidental , special or punitive damages , including , without limitation , loss of profits , whether incurred directly or indirectly , or any loss of data , use , goodwill , or other intangible losses , resulting from : ( i ) your access to or use of or inability to access or use the services , ( ii ) the conduct or content of other users or third parties on , through , or following use of the services ; or ( iii ) unauthorized access , use or alteration of your content , even if tinder has been advised of the possibility of such damages . 



## 3. Loading the LLM
Loads Qwen2.5-7B-Instruct, a free open model, compressed to 4-bit so it fits on Colab's free GPU. A quick test checks that it answers.

In [3]:
!pip install -q bitsandbytes accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

llm_name = "Qwen/Qwen2.5-7B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
    device_map="auto",
)

def ask_llm(prompt, max_new_tokens=200):
    """Send one message to the LLM and return its answer."""
    messages = [{"role": "user", "content": prompt}]
    text = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tok(text, return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)   # no randomness
    return llm_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(ask_llm("Say hello in one short sentence."))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.6 MB/s eta 0:00:00


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Hello!


## 4. Extractive summaries
Asks the LLM for an extractive summary of each long sentence: the most important exact phrases, copied word for word. Each phrase is checked to make sure it appears exactly in the original, then the phrases are joined in their original order with "...". The result must fit within 128 tokens. If it fails, the LLM tries again with a smaller word limit (70, then 55, then 40 words).

In [8]:
import re

PROMPT = """Make an extractive summary of this clause from an online Terms of Service.

List the most important phrases from the clause, one per line.
Rules:
- Copy each phrase exactly as it appears in the clause. Never change, reword or add words.
- Choose the phrases that say what the company or the user can or cannot do: rights, obligations, limits on liability, changes to the terms, account termination, content removal, how disputes are handled, and which laws or courts apply.
- Skip examples, long lists and background.
- All phrases together must be at most {limit} words.
- Return only the phrases, one per line, without numbers or quotation marks.

Example:
Clause: we may , at our sole discretion and for any reason , including but not limited to violations of these terms or suspected fraud , suspend or terminate your account at any time without notice .
Phrases:
we may , at our sole discretion and for any reason
suspend or terminate your account at any time without notice

Clause: {clause}
Phrases:"""

def words(s):
    """Just the words, lowercase, without punctuation."""
    return re.findall(r"[a-z0-9]+", s.lower())

def to_dataset_style(s):
    """Format like the dataset: lowercase, spaces around punctuation, "'s" split off."""
    s = s.lower().strip()
    s = re.sub(r"(\w)'s\b", r"\1 's", s)
    s = re.sub(r"\s*([,;:()])\s*", r" \1 ", s)
    s = re.sub(r"\s*\.$", " .", s)
    return re.sub(r"\s+", " ", s).strip()

def shorten(text):
    """Ask for exact phrases (70, then 55, then 40 words), keep only phrases found exactly in the text."""
    for limit in [70, 55, 40]:
        answer = ask_llm(PROMPT.format(limit=limit, clause=text), max_new_tokens=250)
        lines = [l for l in answer.splitlines() if l.strip()]
        found = []
        for line in lines:
            phrase = to_dataset_style(line.strip(' "-*•'))
            pos = text.find(phrase)                        # where it appears in the original
            if len(words(phrase)) >= 2 and pos != -1 and phrase not in [p for _, p in found]:
                found.append((pos, phrase))
        if not found:
            continue
        found.sort()                                       # back into original order
        summary = " ... ".join(p for _, p in found)
        if len(words(summary)) >= 5 and len(tok(summary)["input_ids"]) <= 128:
            return summary, f"llm ({limit} words), kept {len(found)} of {len(lines)} phrases"
    return None, "failed"

# Test on the first 3 long sentences
for split, i, t in long_items[:3]:
    short, method = shorten(t)
    print(f"{split} #{i} | method: {method}")
    if short:
        print(f"Tokens: {len(tok(t)['input_ids'])} -> {len(tok(short)['input_ids'])}")
        print("SUMMARY:", short)
    print()

train #164 | method: failed

train #234 | method: llm (70 words), kept 5 of 6 phrases
Tokens: 145 -> 116
SUMMARY: you release us from any and all liability in connection with the use of such information by other parties ... we are not responsible for , and can not control , the use by others of any information which you provide to them ... you should use caution in selecting the personal information you provide to others through the service ... we can not assume any responsibility for the content of messages sent by other users of the service ... you release us from any and all liability in connection with the contents of any communications you may receive from other users

train #493 | method: failed



## 4a. Checking why summaries fail
Shows the LLM's raw answer for one sentence at each word limit, and which check fails, to find what needs fixing.

In [7]:
t = long_items[0][2]

for limit in [70, 55, 40]:
    raw = ask_llm(PROMPT.format(limit=limit, clause=t))
    s = to_dataset_style(raw)
    print(f"===== limit {limit} words =====")
    print("RAW ANSWER:", raw)
    print("Words:", len(words(s)), "| Tokens:", len(tok(s)["input_ids"]),
          "| Same words, same order:", is_extractive(s, t))
    orig = iter(words(t))
    for w in words(s):
        if w not in orig:
            print("First word not found in order:", w)
            break
    print()

===== limit 70 words =====
RAW ANSWER: to the fullest extent permitted by law, tinder and its affiliates, employees, licensors, and service providers are not liable for indirect, consequential, incidental, special, or punitive damages, including loss of profits, data, use, or goodwill, from accessing, using, or inability to use the services, conduct of others, or unauthorized access.
Words: 50 | Tokens: 76 | Same words, same order: False
First word not found in order: and

===== limit 55 words =====
RAW ANSWER: to the fullest extent permitted by law, tinder and its affiliates, employees, licensors, and service providers are not liable for indirect, consequential, incidental, special, or punitive damages, including loss of profits, data, use, or goodwill, from accessing, using, or inability to use services, conduct of others, or unauthorized access.
Words: 49 | Tokens: 75 | Same words, same order: False
First word not found in order: and

===== limit 40 words =====
RAW ANSWER: to the fu